In [ ]:
!pip install -q -U transformers sentence-transformers chromadb langchain-text-splitters pypdf sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import torch

from google.colab import files

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
print("Loading FLAN-T5 model...")

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("✅ FLAN-T5 loaded successfully!")

Loading FLAN-T5 model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ FLAN-T5 loaded successfully!


In [ ]:
print("Please upload your PDF:")

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print("✅ Uploaded:", pdf_filename)

Please upload your PDF:


Saving EIC Assignment Question.pdf to EIC Assignment Question.pdf
✅ Uploaded: EIC Assignment Question.pdf


In [ ]:
reader = PdfReader(pdf_filename)

pdf_text = ""

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text:
        pdf_text += text + "\n"

print("✅ PDF loaded successfully!")

print("Number of pages:", len(reader.pages))

print("Characters extracted:", len(pdf_text))

✅ PDF loaded successfully!
Number of pages: 1
Characters extracted: 2251


In [ ]:
print(pdf_text[:2000])

Assignment
 
1:
 
Constitution
 
and
 
Preamble
 
1.
 
Define
 
the
 
Constitution
 
of
 
India
 
and
 
state
 
its
 
primary
 
importance.
 
2.
 
Write
 
a
 
short
 
note
 
on
 
the
 
historical
 
background
 
of
 
the
 
Indian
 
Constitution.
 
3.
 
List
 
any
 
four
 
salient
 
features
 
of
 
the
 
Constitution
 
of
 
India.
 
4.
 
What
 
is
 
the
 
Preamble,
 
and
 
what
 
key
 
values
 
does
 
it
 
highlight?
 
5.
 
Briefly
 
explain
 
the
 
concept
 
of
 
the
 
"Basic
 
Structure"
 
of
 
the
 
Indian
 
Constitution.
 
Assignment
 
2:
 
Fundamental
 
Rights
 
and
 
Directive
 
Principles
 
1.
 
List
 
the
 
six
 
Fundamental
 
Rights
 
guaranteed
 
by
 
Part
 
III
 
of
 
the
 
Constitution.
 
2.
 
State
 
any
 
four
 
Fundamental
 
Duties
 
of
 
an
 
Indian
 
citizen
 
under
 
Part
 
IV-A.
 
3.
 
What
 
is
 
the
 
main
 
objective
 
of
 
the
 
Directive
 
Principles
 
of
 
State
 
Policy
 
(DPSP)?
 
4.
 
Mention
 
two
 
differences
 
between
 
Fundamental
 
Rights
 
and
 
Directi

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(pdf_text)

print("✅ PDF split successfully!")

print("Number of chunks:", len(chunks))

✅ PDF split successfully!
Number of chunks: 3


In [ ]:
print("Loading embedding model...")

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="pdf_chatbot"
)

print("✅ ChromaDB collection created!")

✅ ChromaDB collection created!


In [ ]:
print("Creating embeddings...")

embeddings = embedder.encode(
    chunks
).tolist()

ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings
)

print("✅ PDF chunks stored successfully!")

print("Total chunks:", collection.count())

Creating embeddings...
✅ PDF chunks stored successfully!
Total chunks: 3


In [ ]:
def retrieve_pdf_context(query, top_k=3):

    query_embedding = embedder.encode(
        [query]
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results["documents"][0]

In [ ]:
def generate_answer(question, context):

    prompt = f"""
Answer the question using only the information given below.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
def ask_pdf(query):

    # Retrieve relevant PDF chunks
    context_passages = retrieve_pdf_context(
        query,
        top_k=3
    )

    # Combine retrieved chunks
    context_str = "\n\n".join(
        context_passages
    )

    # Generate answer using FLAN-T5
    answer = generate_answer(
        query,
        context_str
    )

    return answer, context_passages

In [ ]:
answer, context = ask_pdf(
    "What is this PDF about?"
)

print("\n" + "=" * 60)

print("RETRIEVED PDF SNIPPETS")

print("=" * 60)

for i, snippet in enumerate(context, 1):

    print(f"\n[{i}]")
    print(snippet[:300])

print("\n" + "=" * 60)

print("FLAN-T5 RESPONSE")

print("=" * 60)

print(answer)


RETRIEVED PDF SNIPPETS

[1]
Part
 
IV-A.
 
3.
 
What
 
is
 
the
 
main
 
objective
 
of
 
the
 
Directive
 
Principles
 
of
 
State
 
Policy
 
(DPSP)?
 
4.
 
Mention
 
two
 
differences
 
between
 
Fundamental
 
Rights
 
and
 
Directive
 
Principles
 
of
 
State
 
Policy.
 
5.
 
Identify
 
the
 
importance
 
of
 
Fundamental
 

[2]
the
 
purpose
 
of
 
the
 
86th
 
Constitutional
 
Amendment
 
Act
 
(Right
 
to
 
Education).
 
Assignment
 
4:
 
Electoral
 
Literacy
 
and
 
Voter's
 
Education
 
1.
 
What
 
are
 
electoral
 
rights
 
in
 
a
 
democracy?
 
2.
 
List
 
the
 
basic
 
steps
 
required
 
for
 
an
 
eligible
 
citize

[3]
Assignment
 
1:
 
Constitution
 
and
 
Preamble
 
1.
 
Define
 
the
 
Constitution
 
of
 
India
 
and
 
state
 
its
 
primary
 
importance.
 
2.
 
Write
 
a
 
short
 
note
 
on
 
the
 
historical
 
background
 
of
 
the
 
Indian
 
Constitution.
 
3.
 
List
 
any
 
four
 
salient
 
features
 
of
 
th

FLAN-T5 RESPONSE
Disciplinary Principles of State Policy


In [ ]:
print("=" * 60)
print("PDF CHATBOT READY!")
print("Type your question below.")
print("Type 'exit' to quit.")
print("=" * 60)

while True:

    user_query = input(
        "\nAsk a question about your PDF: "
    )

    if user_query.lower().strip() in [
        "exit",
        "quit",
        "q"
    ]:

        print("Exiting PDF Chatbot. Goodbye!")

        break

    if not user_query.strip():
        continue

    try:

        answer, context = ask_pdf(
            user_query
        )

        print("\n--- RETRIEVED PDF SNIPPETS ---")

        for i, snippet in enumerate(
            context,
            1
        ):

            print(
                f"[{i}] {snippet[:150]}..."
            )

        print("\n--- FLAN-T5 RESPONSE ---")

        print(answer)

        print("-" * 60)

    except Exception as e:

        print("\n❌ ERROR:")

        print(e)

        print("-" * 60)

PDF CHATBOT READY!
Type your question below.
Type 'exit' to quit.

--- RETRIEVED PDF SNIPPETS ---
[1] Part
 
IV-A.
 
3.
 
What
 
is
 
the
 
main
 
objective
 
of
 
the
 
Directive
 
Principles
 
of
 
State
 
Policy
 
(DPSP)?
 
4.
 
Mention
 
two
 
diff...
[2] Assignment
 
1:
 
Constitution
 
and
 
Preamble
 
1.
 
Define
 
the
 
Constitution
 
of
 
India
 
and
 
state
 
its
 
primary
 
importance.
 
2.
 
Wri...
[3] the
 
purpose
 
of
 
the
 
86th
 
Constitutional
 
Amendment
 
Act
 
(Right
 
to
 
Education).
 
Assignment
 
4:
 
Electoral
 
Literacy
 
and
 
Voter'...

--- FLAN-T5 RESPONSE ---
The Directive Principles of State Policy.
------------------------------------------------------------
